In [2]:
import torch


In [3]:
import math
import torch.nn as nn


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


Using device: cuda


In [6]:
import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "tiny_shakespeare.txt"
)
print("Downloaded!")


Downloaded!


In [7]:
text = ""
# with open("file.txt", 'r', encoding='utf-8') as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         content = line.split(sep = '-', maxsplit=1)
#         if '<Media omitted>' in content:
#             continue
#         if len(content)<2:
#             text += "\n" + line
#             continue
#         text += "\n"
#         text += content[1].strip()

with open("tiny_shakespeare.txt", 'r', encoding='utf-8') as f:
    for line in f:
        text += line

chars = list(set(text))
chars.sort()
stoi = {char : ind for ind, char in enumerate(chars)}
itos = {ind : char for ind, char in enumerate(chars)}
vocab_size = len(stoi)

def encode(text):
    return [stoi[char] for char in text]

def decode(embedding):
    return [itos[ind] for ind in embedding]

def train_test_split(data, train=0.9):
    n = int(train*len(data))
    tr = data[:n]
    val = data[n:len(data)]
    return tr, val


In [8]:
print(text[:1000])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [23]:
data = torch.tensor(encode(text), dtype=torch.long)
train_data, val_data = train_test_split(data)

torch.manual_seed(42)
batch_size = 32
block_size = 128

def get_batch(split):
    data = train_data if split=='train' else val_data
    inds = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in inds]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in inds]).to(device)
    return x, y

xb, yb = get_batch('train')
print(xb)
print(yb)


tensor([[56, 42,  6,  ..., 53, 51, 43],
        [47, 50, 50,  ..., 43,  1, 51],
        [ 1, 39, 52,  ..., 51, 59, 57],
        ...,
        [ 6,  1, 39,  ..., 43,  6,  1],
        [12,  0,  0,  ..., 25, 27, 28],
        [52, 10,  0,  ...,  0, 32, 46]], device='cuda:0')
tensor([[42,  6,  0,  ..., 51, 43, 42],
        [50, 50,  1,  ...,  1, 51, 53],
        [39, 52, 42,  ..., 59, 57, 58],
        ...,
        [ 1, 39, 52,  ...,  6,  1, 52],
        [ 0,  0, 16,  ..., 27, 28, 31],
        [10,  0, 14,  ..., 32, 46, 43]], device='cuda:0')


In [24]:
# for i in range(batch_size):
#     for j in range(block_size):
#         context = xb[i, :j+1]
#         target = yb[i, j]


In [25]:
class AdamOptimizer:
    def __init__(self, params, lr=3e-4, eps=1e-8, betas=(0.9, 0.999)):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.beta1 = betas[0]
        self.beta2 = betas[1]
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                self.m[i] = self.beta1*self.m[i] + (1-self.beta1)*p.grad
                m_hat = self.m[i]/(1-self.beta1**self.t)
                self.v[i] = self.beta2*self.v[i] + (1-self.beta2)*(p.grad**2)
                v_hat = self.v[i]/(1-self.beta2**self.t)
                p -= (m_hat*self.lr)/(v_hat.sqrt()+self.eps)
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [26]:
def softmax(x):
    exp_x = torch.exp(x)
    return exp_x/exp_x.sum(dim=-1, keepdim=True) # why -1? -> the matrix is batchxqueryxkey, so we have to softmax over all keys

def causal_softmax(x): # matrix = (batch, block_size, block_size) - attention matrix
    mask = torch.tril(torch.ones(block_size, block_size)).to(device)
    x_mask = x.masked_fill(mask == 0, -float('inf'))
    return softmax(x_mask)  


In [34]:
n_embd = 128
d_k = n_embd # Dimension of q, k, v output
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_q = nn.Linear(n_embd, d_k, bias=False)
        self.W_k = nn.Linear(n_embd, d_k, bias=False)
        self.W_v = nn.Linear(n_embd, d_k, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))) # register_buffer - tells pytorch that it is not trainable
    
    def forward(self, batch): # batch = (batch_size, block_size, embedding_size), W_q = (embedding_size, output_size=d_k)
        Q = self.W_q(batch) # (batch_size, block_size, d_k)
        K = self.W_k(batch)
        V = self.W_v(batch)
        w = Q@K.transpose(-2, -1) # we don't transpose each block in a batch, only within each block
        w = w/math.sqrt(d_k)
        # W = causal_softmax(w2) # (batch_size, block_size, block_size)
        w = w.masked_fill(self.mask == 0, -float('inf'))
        W = torch.softmax(w, dim=2) # w - (batch_size, block_size, block_size) - we softmax across third dim
        new_tokens = W @ V
        return new_tokens


In [28]:
def relu(x): return torch.clamp(x, min=0)
def gelu(x): return x*0.5*(1+torch.erf(x/math.sqrt(2)))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 =  nn.Linear(n_embd, 4*n_embd, bias=True)
        self.l2 = nn.Linear(4*n_embd, n_embd, bias=True)

    def forward(self, batch):
        x1 = self.l1(batch)
        x2 = gelu(x1)
        x3 = self.l2(x2)
        return x3


In [29]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = Attention()
        self.attention_norm = nn.LayerNorm(n_embd)
        self.ff = FeedForward()
        self.ff_norm = nn.LayerNorm(n_embd)
        self.dropout = nn.Dropout(0.2) # zeroes out some of the activations (tensor values)
    
    def forward(self, batch):
        batch = batch + self.dropout(self.attention(self.attention_norm(batch))) # nn.Module runs self.forward automatically
        batch = batch + self.dropout(self.ff(self.ff_norm(batch)))
        return batch
    

In [30]:
def loss(logits, y):
    loss_fn = nn.CrossEntropyLoss()
    return loss_fn(logits.view(-1, vocab_size), y.view(-1))


In [32]:
class Model(nn.Module):
    def __init__(self, layers=4):
        super().__init__()
        self.n_layers = layers
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block() for _ in range(self.n_layers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=True)
    
    def forward(self, batch): # batch = [[8 numbers], [], [], []] - 4 batches
        tokens = self.token_embedding(batch) # (batch_size, block_size, n_embd)
        tokens = tokens + self.position_embedding(torch.arange(block_size).to(device)) # (batch, block, n_embd) + (block, n_embd)
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.ln_f(tokens)
        logits = self.lm_head(tokens)
        return logits
    



In [38]:
def train_step(model : Model, optimizer : AdamOptimizer):
    model.train()
    optimizer.zero_grad()
    xb, yb = get_batch('train')
    logits = model(xb)
    step_loss = loss(logits, yb)
    step_loss.backward()
    optimizer.step()
    return step_loss.item()

def val_step(model):
    model.eval()
    with torch.no_grad():
        xb, yb = get_batch('val')
        logits = model(xb)
        step_loss = loss(logits, yb)
    return step_loss.item()

def accuracy(model, samples=1000):
    ...

def epoch(model, optimizer):
    val_interval = 100
    steps = len(train_data)//(batch_size*block_size)
    net_tr_loss = 0
    net_val_loss = 0
    for i in range(1, steps+1):
        tr_loss = train_step(model, optimizer)
        net_tr_loss += tr_loss
        if i%val_interval==0:
            val_loss = val_step(model)
            net_val_loss += val_loss
    return net_tr_loss/steps, net_val_loss/(steps//val_interval)

def train(epochs=1):
    model = torch.compile(Model(layers=4).to(device))
    # optimizer = AdamOptimizer(model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    for n_epoch in range(1, epochs+1):
        tr_loss, val_loss = epoch(model, optimizer)
        print(f"Epoch {n_epoch} | Train Loss: {tr_loss} | Val Loss: {val_loss}")
    return model

model = train(epochs=30)


Epoch 1 | Train Loss: 2.764999024722041 | Val Loss: 2.6155914068222046
Epoch 2 | Train Loss: 2.4600848577460463 | Val Loss: 2.443748712539673
Epoch 3 | Train Loss: 2.2931654764681446 | Val Loss: 2.284487009048462
Epoch 4 | Train Loss: 2.1608637926529863 | Val Loss: 2.18899405002594
Epoch 5 | Train Loss: 2.0791445839161775 | Val Loss: 2.1009130477905273
Epoch 6 | Train Loss: 2.012288237591179 | Val Loss: 2.07857084274292
Epoch 7 | Train Loss: 1.9577934605734688 | Val Loss: 2.014228940010071
Epoch 8 | Train Loss: 1.9069915061094322 | Val Loss: 1.9493115544319153
Epoch 9 | Train Loss: 1.8670539564015913 | Val Loss: 1.9062215089797974
Epoch 10 | Train Loss: 1.8291962083505124 | Val Loss: 1.912581443786621
Epoch 11 | Train Loss: 1.7931606307321666 | Val Loss: 1.8750109672546387
Epoch 12 | Train Loss: 1.7636145645258379 | Val Loss: 1.8036377429962158
Epoch 13 | Train Loss: 1.7366770471845354 | Val Loss: 1.8515029549598694
Epoch 14 | Train Loss: 1.7183713397201226 | Val Loss: 1.82817119359970

In [ ]:
# def generate(model, max_new_tokens=300, temperature=1):
#     model.eval()
#     idx = torch.zeros((1, 1), dtype=torch.long).to(device)
#     for _ in range(max_new_tokens):
#         # pad or crop to block_size
#         if idx.shape[1] < block_size:
#             pad = torch.zeros(1, block_size - idx.shape[1], dtype=torch.long).to(device)
#             idx_cond = torch.cat([pad, idx], dim=1)
#         else:
#             idx_cond = idx[:, -block_size:]
#         logits = model(idx_cond)
#         logits = logits[:, -1, :] / temperature  # last token, apply temperature
#         probs = torch.softmax(logits, dim=-1)
#         next_token = torch.multinomial(probs, num_samples=1)
#         idx = torch.cat([idx, next_token], dim=1)
#     return ''.join(decode(idx[0].tolist()))

# print(generate(model))



ALTHAMISTIUS:
BepOLINGBROKE:
Should ham?

CORIOLANUS:
And thou for throughtness
In to be touch of thee friend of my foes!

GLOUCESTER:
Why, my life thy very mind me known innoce;
No raquain, here some aught am
Plusio, you lords, that is a harts the Cock, hot even
Which mirstage away'd to are you had
